In [1]:
import pickle
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from XReason.xgbooster import XGBooster
# from sklearn.feature_extraction import onehotencoder
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

from keras.losses import SparseCategoricalCrossentropy
from keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.utils import to_categorical
from keras.datasets import mnist
from keras.losses import CategoricalCrossentropy
# from keras.optimizers import Adam
import tf2onnx
import tensorflow as tf
from keras.optimizers.legacy import Adam

/Users/lkiern/XAI_Framework/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv('../dog_data/dog_adoption_master.csv')

In [3]:
y = df['returned']
y_extra = df[['return_reason','days_to_return']]
X = df.drop(['returned','return_reason','days_to_return','adoption_id'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)


X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_test_enc = pd.get_dummies(X_test, drop_first=False)

print('Training set shape: ', np.shape(X_train_enc))
print(f'- Returned:\t {len(y_train[y_train==1])}')
print(f'- Kept: {len(y_train[y_train==0])}')
print('Test set shape: ', np.shape(X_test_enc))
print(f'- Returned:\t {len(y_test[y_test==1])}')
print(f'- Kept: {len(y_test[y_test==0])}')

Training set shape:  (31500, 44)
- Returned:	 4747
- Kept: 26753
Test set shape:  (10500, 44)
- Returned:	 1550
- Kept: 8950


In [5]:
X_test_enc.to_csv('./dog_data_ohe.csv')

In [4]:
X_np = np.array(X_train_enc, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int32)  # Ensure it's an integer array

X_test_np = X_test_enc.to_numpy()
X_test_np = X_test_np.astype('float32')
y_test_np = np.array(y_test, dtype=np.int32)

model_name = 'shelter_returns'
model = Sequential(name=model_name)
model.add(Flatten(input_shape=(44,)))
model.add(Dense(10, activation='relu'))
model.add(Dense(10))
model.add(Dense(2))
model.summary()
model.compile(loss=SparseCategoricalCrossentropy(from_logits=True),
              optimizer=Adam(learning_rate=0.01),
              metrics=['accuracy'])
model.fit(X_np, y_train,
          batch_size=64,
          epochs=3,
          verbose=1)

model_proto, _ = tf2onnx.convert.from_keras(model, output_path='models/' + model_name + '.onnx')



Model: "shelter_returns"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 44)                0         
                                                                 
 dense (Dense)               (None, 10)                450       
                                                                 
 dense_1 (Dense)             (None, 10)                110       
                                                                 
 dense_2 (Dense)             (None, 2)                 22        
                                                                 
Total params: 582 (2.27 KB)
Trainable params: 582 (2.27 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/3
493/493 [==============================] - 0s 346us/step - loss: 0.3973 - accuracy: 0.8487
Epoch 2/3
493/493 [==============================] -

In [5]:
from VeriX_FFA3 import *

Instructions for updating:
non-resource variables are not supported in the long term


In [6]:
X_train_enc.columns

Index(['age_years', 'weight_kg', 'days_in_shelter', 'previously_returned',
       'neutered', 'aggression_score', 'anxiety_separation',
       'reactivity_to_dogs', 'energy_level', 'training_level', 'house_trained',
       'first_time_owner', 'household_has_kids', 'household_has_pets',
       'has_yard', 'hours_alone_per_day', 'adopter_activity_level',
       'visits_before_adoption', 'met_resident_pets', 'adoption_counseling',
       'expectation_score', 'energy_mismatch', 'size_home_mismatch',
       'size_large', 'size_medium', 'size_small', 'size_xlarge',
       'breed_group_herding', 'breed_group_hound', 'breed_group_mixed',
       'breed_group_sporting', 'breed_group_terrier', 'breed_group_toy',
       'breed_group_working', 'intake_type_born_in_care',
       'intake_type_owner_surrender', 'intake_type_stray',
       'intake_type_transfer', 'medical_needs_chronic', 'medical_needs_minor',
       'medical_needs_none', 'home_type_apartment', 'home_type_house_owned',
       'home_typ

In [7]:
categorical_values_dict = {
    "size_small":[0,1],
    "size_medium":[0,1],
    "size_large":[0,1],
    "size_xlarge":[0,1],
    "breed_group_hound":[0,1],
    "breed_group_terrier":[0,1],
    "breed_group_herding":[0,1],
    "breed_group_working":[0,1],
    "breed_group_toy":[0,1],
    "breed_group_sporting":[0,1],
    "breed_group_mixed":[0,1],
    "intake_type_stray":[0,1],
    "intake_type_owner_surrender":[0,1],
    "intake_type_transfer":[0,1],
    "intake_type_born_in_care":[0,1],
    "previously_returned":[0,1],
    "medical_needs_none":[0,1],
    "medical_needs_minor":[0,1],
    "medical_needs_chronic":[0,1],
    "neutered":[0,1],
    "house_trained": [0,1],
    "first_time_owner": [0,1],
    "household_has_kids": [0,1],
    "household_has_pets" : [0,1],
    "home_type_house_owned":[0,1],
    "home_type_house_rented":[0,1],
    "home_type_apartment":[0,1],
    "has_yard": [0,1],
    "adoption_counseling": [0,1],
    "met_resident_pets": [0,1]
}

In [8]:
one_hot_groups=[["size_small", "size_medium", "size_large","size_xlarge"],
                ["breed_group_hound","breed_group_terrier","breed_group_herding","breed_group_working","breed_group_toy","breed_group_sporting","breed_group_mixed"],
                ["intake_type_stray","intake_type_owner_surrender","intake_type_transfer","intake_type_born_in_care"],
                ["medical_needs_none","medical_needs_minor","medical_needs_chronic"],
                ["home_type_house_owned","home_type_house_rented","home_type_apartment"],]

In [9]:
v3a = VeriX(
    datatype="tabular",
    feature_names=X_test_enc.columns.tolist(),
    model_path="models/shelter_returns.onnx",
    plot_original=False,
    categorical_values_dict=categorical_values_dict,
    one_hot_groups=one_hot_groups,
    time_limit=90,
    in_jupyter = True,
    stakeholder_name='developer'
)

In [10]:
v3a.explain(X_test_np[3], epsilon=5 , plot_explanation=False, prediction_label="Rescue Failed")


Relevant features (sat): []
Irrelevant features (unsat): [43]
Timeout features: []
n:44
exiting enumeration
sat_set: []
expl_strings: {'abd': ['IF TRUE THEN label = 0', 'IF age_years = 2.0 THEN label = 0', 'IF aggression_score = 2.0999999046325684 AND visits_before_adoption = 2.0 AND size_home_mismatch = 1.0 THEN label = 0', 'IF aggression_score = 2.0999999046325684 AND energy_mismatch = 2.299999952316284 AND size_home_mismatch = 1.0 THEN label = 0', 'IF visits_before_adoption = 2.0 AND expectation_score = 6.199999809265137 AND energy_mismatch = 2.299999952316284 AND size_home_mismatch = 1.0 THEN label = 0'], 'con': ['IF age_years = -3.0 AND aggression_score = 6.81509 AND training_level = -1.9 AND visits_before_adoption = -3.0 AND energy_mismatch = -2.7 THEN prediction != 0', 'IF age_years = -3.0 AND aggression_score = 6.24961 AND training_level = -1.9 AND energy_mismatch = -2.7 AND size_home_mismatch = 6.0 THEN prediction != 0', 'IF age_years = -3.0 AND aggression_score = 5.91403 AND 

Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…